## 💰 Project Overview: Budget Buddy

Welcome to my Smart Finance Assistant project. This notebook develops **Budget Buddy**, a simple AI-powered personal finance assistant that helps users understand their spending and make better budgeting decisions.

Final Application Components:

💬 **AI Chat Interface:** A friendly Budget Buddy chatbot that gives spending advice  
📊 **Data Analysis:** CSV transaction analysis for expenses, refunds, net spending, and budget status  
🔍 **RAG System:** Simple retrieval from financial advice documents  
🛠️ **Custom Tool:** A savings goal calculator for planning future savings  
🌐 **Gradio UI:** A basic web interface that connects all features together  

**Development Approach:**  
The project is built step by step, starting from CSV processing and spending analysis, then adding AI chat, document retrieval, custom tools, testing, and a final Gradio interface.

---

# 🚀 Getting Started: Foundation Setup

## Initial Setup
This cell installs the necessary libraries.

In [26]:
# Install required packages when running in Google Colab
!pip install gradio pandas hands-on-ai

# Import core libraries
import pandas as pd
import numpy as np
import gradio as gr
from datetime import datetime
import warnings

warnings.filterwarnings("ignore")

print("Core libraries loaded successfully!")
print(f"Pandas version: {pd.__version__}")

Core libraries loaded successfully!
Pandas version: 2.2.2


## Hands-on-AI Configuration

Set up the hands-on-ai package for advanced features (chat, RAG, tools):

In [27]:
import os

# Configure hands-on-ai server connection
os.environ['HANDS_ON_AI_SERVER'] = 'https://ollama.locollm.org'
os.environ['HANDS_ON_AI_MODEL'] = 'gemma3:4b'
os.environ['HANDS_ON_AI_API_KEY'] = 'Curtin2026ISYS20015002'

print("🔑 Hands-on-AI configured successfully!")

🔑 Hands-on-AI configured successfully!


## Connection Test

Test that everything is working correctly:

In [28]:
from hands_on_ai.chat import get_response

# Test the connection to the hands-on-ai server
try:
    response = get_response("Hello! I'm building a Smart Finance Assistant.")
    print("✅ Hands-on-AI connection successful!")
    print(f"Response: {response}")
except Exception as e:
    print(f"❌ Connection issue: {e}")
    print("You can still work on the data processing foundation without this.")

✅ Hands-on-AI connection successful!
Response: That’s fantastic! A Smart Finance Assistant – that’s a really interesting and useful project. I’m excited to hear about it. 

To help me understand how I can best assist you with building it, could you tell me a bit more about your project? For example:

*   **What's the overall goal of your Smart Finance Assistant?** (e.g., helping users track spending, budgeting, investing, debt management, etc.)
*   **What platforms are you targeting?** (e.g., web application, mobile app (iOS, Android), voice assistant integration - Alexa, Google Assistant?)
*   **What kind of features are you planning to include?** (e.g., transaction categorization, goal setting, reports, alerts, financial advice?)
*   **What's your technical background/experience?** (e.g., are you a solo developer, part of a team, familiar with specific programming languages/frameworks?)

Don't worry if you don’t have all the answers right now – this is just a starting point.  Let’s f

# 🏗️ Foundation: Data Processing Skills


## Sample Transaction Data Setup

In [29]:
# Sample data structure for testing Budget Buddy
# Note: Amount includes $ signs and one refund to test data cleaning later

sample_transactions = {
    'Date': [
        '2026-05-01', '2026-05-02', '2026-05-03', '2026-05-04', '2026-05-05',
        '2026-05-06', '2026-05-07', '2026-05-08', '2026-05-09', '2026-05-10',
        '2026-05-11', '2026-05-12', '2026-05-13', '2026-05-14', '2026-05-15',
        '2026-05-16', '2026-05-17', '2026-05-18', '2026-05-19', '2026-05-20',
        '2026-05-21'
    ],
    'Description': [
        'Groceries', 'Coffee', 'Lunch', 'Bus ticket', 'Train ticket',
        'Uber ride', 'Netflix', 'Spotify', 'Phone bill', 'Internet bill',
        'Clothes', 'Shoes', 'Medicine', 'Gym', 'Movie ticket',
        'Restaurant dinner', 'Takeaway', 'Coffee', 'Groceries', 'Bus ticket',
        'Returned item'
    ],
    'Category': [
        'Food', 'Food', 'Food', 'Transport', 'Transport',
        'Transport', 'Subscription', 'Subscription', 'Bills', 'Bills',
        'Shopping', 'Shopping', 'Health', 'Health', 'Entertainment',
        'Dining', 'Dining', 'Food', 'Food', 'Transport',
        'Refund'
    ],
    'Amount': [
        '$45.20', '$6.50', '$14.00', '$4.80', '$5.20',
        '$22.00', '$18.99', '$12.99', '$40.00', '$65.00',
        '$55.00', '$80.00', '$18.50', '$30.00', '$19.50',
        '$48.00', '$32.50', '$6.00', '$52.40', '$4.80',
        '-$25.00'
    ]
}

# Create DataFrame from sample data
df_sample = pd.DataFrame(sample_transactions)

print("Sample transaction data created:")
print(df_sample)

# Clean Amount column by removing dollar signs
df_sample["Amount"] = (
    df_sample["Amount"]
    .astype(str)
    .str.replace("$", "", regex=False)
)

# Convert Amount column to numeric values
df_sample["Amount"] = pd.to_numeric(df_sample["Amount"], errors="coerce")

print("Cleaned transaction data:")
print(df_sample)

Sample transaction data created:
          Date        Description       Category   Amount
0   2026-05-01          Groceries           Food   $45.20
1   2026-05-02             Coffee           Food    $6.50
2   2026-05-03              Lunch           Food   $14.00
3   2026-05-04         Bus ticket      Transport    $4.80
4   2026-05-05       Train ticket      Transport    $5.20
5   2026-05-06          Uber ride      Transport   $22.00
6   2026-05-07            Netflix   Subscription   $18.99
7   2026-05-08            Spotify   Subscription   $12.99
8   2026-05-09         Phone bill          Bills   $40.00
9   2026-05-10      Internet bill          Bills   $65.00
10  2026-05-11            Clothes       Shopping   $55.00
11  2026-05-12              Shoes       Shopping   $80.00
12  2026-05-13           Medicine         Health   $18.50
13  2026-05-14                Gym         Health   $30.00
14  2026-05-15       Movie ticket  Entertainment   $19.50
15  2026-05-16  Restaurant dinner      

---

# 📊 Six-Step Development Methodology

## STEP 1: Understand the Problem

## Problem Definition

**Problem Statement:**
Budget Buddy helps students and young adults who have limited monthly budgets but do not clearly know where their money is going. By analysing a simple transaction CSV, the app helps them understand their spending, identify their biggest expense categories, notice refunds or repeated costs, and receive simple budgeting advice. The value is that users can make better spending decisions without needing advanced finance knowledge.

## STEP 2: Identify Inputs and Outputs

**Input/Output Definition:**

**Inputs:**
- CSV file
- Date
- Description
- Category
- Amount
- Monthly budget
- User question

**Outputs:**
- Total spending
- Spending by category
- Highest spending category
- Refunds
- Budget status
- Simple budgeting advice
- Chatbot answer based on the CSV data

## STEP 3: Work the Problem by Hand

Before writing Python code, I manually calculated a few examples from the transaction data. This helps me understand what the app should calculate.

### Example 1: Normal Spending — Food Expenses

| Date       | Description | Category | Amount |
|------------|-------------|----------|--------|
| 1/05/2026  | Groceries   | Food     | \$45.20 |
| 2/05/2026  | Coffee      | Food     | \$6.50  |
| 3/05/2026  | Lunch       | Food     | \$14.00 |
| 18/05/2026 | Coffee      | Food     | \$6.00  |
| 19/05/2026 | Groceries   | Food     | \$52.40 |

**Manual calculation:**

Food spending = 45.20 + 6.50 + 14.00 + 6.00 + 52.40  
Food spending = **$124.10**

**Insight:**

The user spent **$124.10 on Food**. This shows that regular expenses like groceries, coffee, and lunch can add up over the month.

---

### Example 2: Refund Transaction

| Date | Description | Category | Amount |
|---|---|---|---:|
| 21/05/2026 | Returned item | Refund | \-$25.00 |

**Manual calculation:**

Total spending before refund = \$581.38  
Refund = \-$25.00  

Total after refund = 581.38 - 25.00  
Total after refund = **$556.38**

**Insight:**

The refund reduced total spending from **\$581.38 to \$556.38**. The app should handle negative amounts correctly so refunds reduce spending instead of increasing it.

---

### Example 3: Large Purchases

| Date | Description | Category | Amount |
|---|---|---|---:|
| 10/05/2026 | Internet bill | Bills | \$65.00 |
| 11/05/2026 | Clothes | Shopping | \$55.00 |
| 12/05/2026 | Shoes | Shopping | \$80.00 |
| 19/05/2026 | Groceries | Food | \$52.40 |

**Manual calculation:**

Large purchases total = 65.00 + 55.00 + 80.00 + 52.40  
Large purchases total = **$252.40**

**Compare with total spending:**

Percentage of spending = 252.40 / 581.38 × 100  
Percentage of spending = **43.4%**

**Insight:**

These four large purchases make up about **43.4% of total spending**. This means a small number of large transactions can have a big impact on the user’s budget.

## STEP 4: Write Pseudocode

Sketch the algorithm in plain English before coding.

```
Step 1: Load and check data
- Read the transaction CSV file.
- Check that Date, Description, Category, and Amount columns exist.
- Check that the CSV is not empty.

Step 2: Clean data
- Remove dollar signs and commas from Amount values.
- Convert Amount values into numbers.
- Keep refunds as negative values.
- Remove or report rows with invalid Amount values.
- Standardise Category names so they are consistent.

Step 3: Analyse spending
- Separate expenses and refunds.
- Calculate total expenses.
- Calculate total refunds.
- Calculate net spending after refunds.
- Group spending by category.
- Find the highest spending category.
- Find the top 3 spending categories.
- Identify large transactions.

Step 4: Check budget
- Ask the user for a monthly budget.
- Check that the budget is greater than zero.
- Compare net spending with the monthly budget.
- Calculate the amount remaining or the amount over budget.
- Calculate the percentage of budget used.

Step 5: Give advice and display results
- Show total expenses, refunds, and net spending.
- Show spending by category.
- Show the highest spending category.
- Show budget status.
- Give simple advice based on the highest spending category.
- Warn the user if they are over budget.
- Show chatbot advice based on the CSV data.
```

## STEP 5: Convert to Python

**💻 Implementation with AI Collaboration**

Now implement your solution using AI assistance. Focus on creating professional, business-appropriate code.


## 🤖 Implementation Strategy

**Effective AI Prompts for Implementation:**
```
"I'm implementing a Smart Finance Assistant. Based on my pseudocode, please create
a Python function that [specific functionality]. The code should:
- Handle real-world CSV data issues (dollar signs, missing values)
- Include clear comments explaining business logic
- Use professional variable names
- Format output for business presentation
- Include basic error handling"
```

**Remember to critique and improve AI responses before using them!**
:::

### Foundation Data Processing Functions

In [30]:
import pandas as pd
import re

def load_and_clean_transaction_data(file_path):
    """
    Load and clean transaction data for the Smart Finance Assistant.

    This function prepares raw CSV transaction data for business analysis by:
    - Validating required columns
    - Cleaning currency values in the Amount column
    - Handling missing values
    - Keeping refunds as negative values
    - Standardising category names
    - Removing rows that cannot be used for financial analysis

    Args:
        file_path: Path to CSV file, uploaded file object, or pandas DataFrame

    Returns:
        pandas.DataFrame: Cleaned transaction data ready for analysis
    """

    required_columns = ["Date", "Description", "Category", "Amount"]

    try:
        # Load data depending on input type.
        # This supports a CSV path, uploaded file object, or an existing DataFrame.
        if isinstance(file_path, pd.DataFrame):
            transaction_data = file_path.copy()
        else:
            transaction_data = pd.read_csv(file_path)

    except FileNotFoundError:
        raise FileNotFoundError(
            "Business data error: The transaction CSV file could not be found. "
            "Please check the file path and try again."
        )

    except Exception as error:
        raise Exception(
            f"Business data error: The transaction file could not be loaded. "
            f"Please check that it is a valid CSV file. Details: {error}"
        )

    # Check that the CSV file is not empty.
    if transaction_data.empty:
        raise ValueError(
            "Business data error: The transaction file is empty. "
            "Please upload a CSV file that contains transaction records."
        )

    # Validate that all required columns are present.
    missing_columns = [
        column for column in required_columns
        if column not in transaction_data.columns
    ]

    if missing_columns:
        raise ValueError(
            "Business data error: The CSV file is missing required columns: "
            + ", ".join(missing_columns)
            + ". Please make sure the file includes Date, Description, Category, and Amount."
        )

    # Keep only the columns needed for this finance assistant.
    cleaned_data = transaction_data[required_columns].copy()

    # Remove rows where all required fields are missing.
    cleaned_data = cleaned_data.dropna(how="all", subset=required_columns)

    # Clean text fields for business presentation.
    cleaned_data["Description"] = (
        cleaned_data["Description"]
        .fillna("No description provided")
        .astype(str)
        .str.strip()
    )

    cleaned_data["Category"] = (
        cleaned_data["Category"]
        .fillna("Uncategorised")
        .astype(str)
        .str.strip()
        .str.title()
    )

    # Convert Date into datetime format.
    # Invalid dates are kept as NaT so they can be reviewed later.
    cleaned_data["Date"] = pd.to_datetime(
        cleaned_data["Date"],
        errors="coerce"
    )

    # Clean Amount values.
    # This handles real-world issues such as "$1,200.50", "-$25.00", missing values,
    # and accounting-style negative numbers like "($50.00)".
    def clean_amount_value(amount):
        if pd.isna(amount):
            return None

        amount_text = str(amount).strip()

        # Convert accounting-style negatives, e.g. ($25.00), into -25.00
        if amount_text.startswith("(") and amount_text.endswith(")"):
            amount_text = "-" + amount_text[1:-1]

        # Remove dollar signs, commas, and spaces.
        amount_text = amount_text.replace("$", "").replace(",", "").replace(" ", "")

        # Keep only valid numeric characters.
        amount_text = re.sub(r"[^0-9.\-]", "", amount_text)

        try:
            return float(amount_text)
        except ValueError:
            return None

    cleaned_data["Amount"] = cleaned_data["Amount"].apply(clean_amount_value)

    # Identify invalid amount rows before removing them.
    invalid_amount_count = cleaned_data["Amount"].isna().sum()

    # Remove rows with invalid Amount values because they cannot be used in financial analysis.
    cleaned_data = cleaned_data.dropna(subset=["Amount"])

    # Remove rows with missing dates because time-based financial analysis requires valid dates.
    invalid_date_count = cleaned_data["Date"].isna().sum()
    cleaned_data = cleaned_data.dropna(subset=["Date"])

    # Sort data by date for a clearer business presentation.
    cleaned_data = cleaned_data.sort_values(by="Date").reset_index(drop=True)

    # Format Category again after cleaning to make labels consistent.
    cleaned_data["Category"] = cleaned_data["Category"].replace({
        "": "Uncategorised",
        "Nan": "Uncategorised",
        "None": "Uncategorised"
    })

    # Display a short business-focused summary for the notebook user.
    print("Transaction data loaded and cleaned successfully.")
    print(f"Cleaned records available for analysis: {len(cleaned_data)}")
    print(f"Rows removed due to invalid Amount values: {invalid_amount_count}")
    print(f"Rows removed due to invalid Date values: {invalid_date_count}")
    print("The data is now ready for spending analysis.")

    return cleaned_data

# Test your function
clean_data = load_and_clean_transaction_data(df_sample)
# print(clean_data)

Transaction data loaded and cleaned successfully.
Cleaned records available for analysis: 21
Rows removed due to invalid Amount values: 0
Rows removed due to invalid Date values: 0
The data is now ready for spending analysis.


In [31]:
# 🤖 AI Collaboration: Spending Analysis and Budget Checking Function
# AI helped create a function to analyse cleaned transaction data,
# calculate spending patterns, check monthly budget status,
# and generate business-friendly financial insights.
def analyze_spending_patterns(df, monthly_budget):
    """
    Analyze spending patterns and budget status for the Finance Buddy.

    This function uses cleaned transaction data to calculate spending totals,
    identify major spending categories, detect refunds, find large transactions,
    and compare net spending against a user-provided monthly budget.

    Args:
        df: Cleaned transaction DataFrame with Date, Description, Category, Amount columns
        monthly_budget: User's monthly budget amount

    Returns:
        dict: Business-friendly analysis results and financial insights
    """

    required_columns = ["Date", "Description", "Category", "Amount"]

    try:
        # Validate that the input is a DataFrame.
        if not isinstance(df, pd.DataFrame):
            raise TypeError(
                "Business analysis error: The transaction data must be a pandas DataFrame."
            )

        # Check that the DataFrame is not empty.
        if df.empty:
            raise ValueError(
                "Business analysis error: The transaction data is empty. "
                "Please load and clean a valid CSV file before running analysis."
            )

        # Check that all required columns exist.
        missing_columns = [
            column for column in required_columns
            if column not in df.columns
        ]

        if missing_columns:
            raise ValueError(
                "Business analysis error: The data is missing required columns: "
                + ", ".join(missing_columns)
            )

        # Validate and convert monthly budget.
        try:
            monthly_budget = float(monthly_budget)
        except ValueError:
            raise ValueError(
                "Business analysis error: Monthly budget must be a valid number."
            )

        if monthly_budget <= 0:
            raise ValueError(
                "Business analysis error: Monthly budget must be greater than zero."
            )

        # Create a copy so the original cleaned data is not changed.
        transaction_data = df.copy()

        # Ensure Amount is numeric before analysis.
        transaction_data["Amount"] = pd.to_numeric(
            transaction_data["Amount"],
            errors="coerce"
        )

        # Remove rows where Amount is invalid, because they cannot support financial analysis.
        transaction_data = transaction_data.dropna(subset=["Amount"])

        if transaction_data.empty:
            raise ValueError(
                "Business analysis error: No valid transaction amounts are available for analysis."
            )

        # Separate expenses and refunds.
        # Business logic: positive amounts are expenses, negative amounts are refunds.
        expense_transactions = transaction_data[transaction_data["Amount"] > 0].copy()
        refund_transactions = transaction_data[transaction_data["Amount"] < 0].copy()

        # Calculate core financial totals.
        total_expenses = expense_transactions["Amount"].sum()
        total_refunds = abs(refund_transactions["Amount"].sum())
        net_spending = total_expenses - total_refunds

        # Group expenses by category for spending analysis.
        category_summary = (
            expense_transactions
            .groupby("Category")["Amount"]
            .sum()
            .reset_index()
            .sort_values(by="Amount", ascending=False)
        )

        # Add percentage of total spending for each category.
        if total_expenses > 0:
            category_summary["Percentage"] = (
                category_summary["Amount"] / total_expenses * 100
            )
        else:
            category_summary["Percentage"] = 0

        # Format category summary for business presentation.
        category_summary["Formatted Amount"] = category_summary["Amount"].apply(
            lambda amount: f"${amount:,.2f}"
        )
        category_summary["Formatted Percentage"] = category_summary["Percentage"].apply(
            lambda percentage: f"{percentage:.1f}%"
        )

        # Find the highest spending category.
        if not category_summary.empty:
            highest_spending_category = category_summary.iloc[0]["Category"]
            highest_spending_amount = category_summary.iloc[0]["Amount"]
        else:
            highest_spending_category = "No expense categories found"
            highest_spending_amount = 0

        # Find the top 3 spending categories.
        top_3_categories = category_summary.head(3)

        # Identify large transactions.
        # Business logic: a large transaction is defined as any expense above the 75th percentile.
        if not expense_transactions.empty:
            large_transaction_threshold = expense_transactions["Amount"].quantile(0.75)
            large_transactions = (
                expense_transactions[expense_transactions["Amount"] >= large_transaction_threshold]
                .sort_values(by="Amount", ascending=False)
            )
        else:
            large_transaction_threshold = 0
            large_transactions = expense_transactions

        # Check budget status.
        amount_remaining = monthly_budget - net_spending
        percentage_budget_used = (net_spending / monthly_budget) * 100

        if net_spending > monthly_budget:
            budget_status = "Over budget"
            budget_message = (
                f"You are over budget by ${abs(amount_remaining):,.2f}."
            )
        else:
            budget_status = "Within budget"
            budget_message = (
                f"You have ${amount_remaining:,.2f} remaining in your monthly budget."
            )

        # Generate simple business-friendly insights.
        insights = []

        insights.append(
            f"Total expenses were ${total_expenses:,.2f}, with refunds of ${total_refunds:,.2f}."
        )

        insights.append(
            f"Net spending after refunds was ${net_spending:,.2f}."
        )

        insights.append(
            f"The highest spending category was {highest_spending_category} "
            f"at ${highest_spending_amount:,.2f}."
        )

        insights.append(
            f"You used {percentage_budget_used:.1f}% of your monthly budget."
        )

        if budget_status == "Over budget":
            insights.append(
                "Budget warning: Your net spending is higher than your monthly budget. "
                "Consider reducing flexible spending categories."
            )
        else:
            insights.append(
                "Budget status: Your spending is currently within the monthly budget."
            )

        if highest_spending_category != "No expense categories found":
            insights.append(
                f"Actionable advice: Review your {highest_spending_category} spending "
                "to see if there are opportunities to save money."
            )

        # Create a business presentation summary.
        presentation_summary = f"""
Finance Buddy - Spending Analysis Summary

Total Expenses: ${total_expenses:,.2f}
Total Refunds: ${total_refunds:,.2f}
Net Spending: ${net_spending:,.2f}

Monthly Budget: ${monthly_budget:,.2f}
Budget Used: {percentage_budget_used:.1f}%
Budget Status: {budget_status}
{budget_message}

Highest Spending Category: {highest_spending_category}
Highest Category Amount: ${highest_spending_amount:,.2f}
Large Transaction Threshold: ${large_transaction_threshold:,.2f}
"""

        # Return both raw values and formatted outputs for later use in Gradio.
        analysis_results = {
            "total_expenses": total_expenses,
            "total_refunds": total_refunds,
            "net_spending": net_spending,
            "monthly_budget": monthly_budget,
            "amount_remaining": amount_remaining,
            "percentage_budget_used": percentage_budget_used,
            "budget_status": budget_status,
            "budget_message": budget_message,
            "highest_spending_category": highest_spending_category,
            "highest_spending_amount": highest_spending_amount,
            "category_summary": category_summary,
            "top_3_categories": top_3_categories,
            "large_transactions": large_transactions,
            "large_transaction_threshold": large_transaction_threshold,
            "insights": insights,
            "presentation_summary": presentation_summary
        }

        return analysis_results

    except Exception as error:
        raise Exception(f"Spending analysis failed: {error}")

# Test your function
analysis = analyze_spending_patterns(clean_data, monthly_budget=2000)

print(analysis["presentation_summary"])

print("Top 3 Spending Categories:")
display(analysis["top_3_categories"])

print("Large Transactions:")
display(analysis["large_transactions"])

print("Financial Insights:")
for insight in analysis["insights"]:
    print("-", insight)


Finance Buddy - Spending Analysis Summary

Total Expenses: $581.38
Total Refunds: $25.00
Net Spending: $556.38

Monthly Budget: $2,000.00
Budget Used: 27.8%
Budget Status: Within budget
You have $1,443.62 remaining in your monthly budget.

Highest Spending Category: Shopping
Highest Category Amount: $135.00
Large Transaction Threshold: $45.90

Top 3 Spending Categories:


,Category,Amount,Percentage,Formatted Amount,Formatted Percentage
5,Shopping,135.0,23.220613,$135.00,23.2%
3,Food,124.1,21.345764,$124.10,21.3%
0,Bills,105.0,18.060477,$105.00,18.1%


Large Transactions:


,Date,Description,Category,Amount
11,2026-05-12,Shoes,Shopping,80.0
9,2026-05-10,Internet bill,Bills,65.0
10,2026-05-11,Clothes,Shopping,55.0
18,2026-05-19,Groceries,Food,52.4
15,2026-05-16,Restaurant dinner,Dining,48.0


Financial Insights:
- Total expenses were $581.38, with refunds of $25.00.
- Net spending after refunds was $556.38.
- The highest spending category was Shopping at $135.00.
- You used 27.8% of your monthly budget.
- Budget status: Your spending is currently within the monthly budget.
- Actionable advice: Review your Shopping spending to see if there are opportunities to save money.


In [32]:
# 🤖 AI Collaboration: Business Insights and Recommendations Generator
# AI helped create a function that turns spending analysis results
# into a clear business-friendly financial report with user advice.
# This supports Step 5 of my pseudocode: Give advice and display results.

def generate_financial_recommendations(analysis_data):
    """
    Generate actionable financial recommendations based on spending analysis.

    This function turns the spending analysis dictionary into a clear,
    user-friendly financial report for the Smart Finance Assistant.

    Args:
        analysis_data: Dictionary with spending analysis results

    Returns:
        str: Formatted recommendations report
    """

    try:
        # Check that the input is a dictionary.
        if not isinstance(analysis_data, dict):
            raise TypeError(
                "Recommendation error: analysis_data must be a dictionary created by analyze_spending_patterns()."
            )

        # Required values from the spending analysis function.
        required_keys = [
            "total_expenses",
            "total_refunds",
            "net_spending",
            "budget_status",
            "budget_message",
            "highest_spending_category",
            "highest_spending_amount",
            "category_summary"
        ]

        # Check whether any required values are missing.
        missing_keys = [
            key for key in required_keys
            if key not in analysis_data
        ]

        if missing_keys:
            raise ValueError(
                "Recommendation error: The analysis data is missing required values: "
                + ", ".join(missing_keys)
            )

        # Extract key financial results.
        total_expenses = analysis_data["total_expenses"]
        total_refunds = analysis_data["total_refunds"]
        net_spending = analysis_data["net_spending"]
        budget_status = analysis_data["budget_status"]
        budget_message = analysis_data["budget_message"]
        highest_spending_category = analysis_data["highest_spending_category"]
        highest_spending_amount = analysis_data["highest_spending_amount"]
        category_summary = analysis_data["category_summary"]

        # Build the spending by category section.
        # This makes the report easier for the user to read.
        category_report_lines = []

        if hasattr(category_summary, "iterrows") and not category_summary.empty:
            for _, row in category_summary.iterrows():
                category_name = row["Category"]
                category_amount = row["Amount"]

                if "Percentage" in category_summary.columns:
                    category_percentage = row["Percentage"]
                    category_report_lines.append(
                        f"- {category_name}: ${category_amount:,.2f} ({category_percentage:.1f}% of total expenses)"
                    )
                else:
                    category_report_lines.append(
                        f"- {category_name}: ${category_amount:,.2f}"
                    )
        else:
            category_report_lines.append(
                "- No category spending data is available."
            )

        category_report = "\n".join(category_report_lines)

        # Generate advice based on the highest spending category.
        # This provides simple chatbot-style guidance linked to the user's CSV data.
        if highest_spending_category != "No expense categories found":
            category_advice = (
                f"Your highest spending category is {highest_spending_category}, "
                f"with ${highest_spending_amount:,.2f} spent. "
                f"Budget Buddy recommends reviewing this category first because it has "
                f"the biggest impact on your overall spending."
            )
        else:
            category_advice = (
                "Budget Buddy could not identify a main spending category. "
                "Please check that the transaction data includes valid expense records."
            )

        # Generate budget advice based on whether the user is over budget.
        if budget_status == "Over budget":
            budget_advice = (
                "⚠️ Budget warning: You are currently over budget. "
                "Consider reducing non-essential spending and reviewing your largest transactions."
            )
        else:
            budget_advice = (
                "✅ Good progress: You are currently within your monthly budget. "
                "Keep monitoring your spending so you can stay on track."
            )

        # Create a chatbot-style recommendation message.
        chatbot_advice = (
            f"Chatbot advice: Based on your CSV transaction data, your net spending is "
            f"${net_spending:,.2f}. {category_advice} {budget_advice}"
        )

        # Format the final report for business presentation.
        recommendations_report = f"""
Smart Finance Assistant - Financial Recommendations Report

1. Spending Summary
Total Expenses: ${total_expenses:,.2f}
Total Refunds: ${total_refunds:,.2f}
Net Spending After Refunds: ${net_spending:,.2f}

2. Spending by Category
{category_report}

3. Highest Spending Category
Highest Category: {highest_spending_category}
Amount Spent: ${highest_spending_amount:,.2f}

4. Budget Status
Status: {budget_status}
{budget_message}

5. Recommendation
{category_advice}

6. Budget Advice
{budget_advice}

"""

        return recommendations_report

    except Exception as error:
        return f"Financial recommendations could not be generated. Details: {error}"

# Test your function
recommendations = generate_financial_recommendations(analysis)

print(recommendations)



Smart Finance Assistant - Financial Recommendations Report

1. Spending Summary
Total Expenses: $581.38
Total Refunds: $25.00
Net Spending After Refunds: $556.38

2. Spending by Category
- Shopping: $135.00 (23.2% of total expenses)
- Food: $124.10 (21.3% of total expenses)
- Bills: $105.00 (18.1% of total expenses)
- Dining: $80.50 (13.8% of total expenses)
- Health: $48.50 (8.3% of total expenses)
- Transport: $36.80 (6.3% of total expenses)
- Subscription: $31.98 (5.5% of total expenses)
- Entertainment: $19.50 (3.4% of total expenses)

3. Highest Spending Category
Highest Category: Shopping
Amount Spent: $135.00

4. Budget Status
Status: Within budget
You have $1,443.62 remaining in your monthly budget.

5. Recommendation
Your highest spending category is Shopping, with $135.00 spent. Budget Buddy recommends reviewing this category first because it has the biggest impact on your overall spending.

6. Budget Advice
✅ Good progress: You are currently within your monthly budget. Keep

---

# 🌐 Advanced Features: Integrating AI Components


## Chat Interface Integration

In [33]:
# 🤖 AI Collaboration: Financial Advice Chatbot
# AI helped create a finance-focused chatbot personality for Budget Buddy.
# The chatbot uses the user's CSV analysis results to give practical advice.

from hands_on_ai.chat import get_response

def create_finance_chat_personality(analysis_data):
    """
    Create a finance-focused chatbot personality using transaction analysis data.

    The chatbot uses results from the user's CSV data, including total expenses,
    refunds, net spending, highest spending category, and budget status.

    Args:
        analysis_data: Dictionary created by analyze_spending_patterns()

    Returns:
        function: A chatbot function that answers user questions
    """

    required_keys = [
        "total_expenses",
        "total_refunds",
        "net_spending",
        "highest_spending_category",
        "budget_status"
    ]

    missing_keys = [
        key for key in required_keys
        if key not in analysis_data
    ]

    if missing_keys:
        raise KeyError(
            "Chatbot setup error: Missing required analysis values: "
            + ", ".join(missing_keys)
        )

    total_expenses = analysis_data["total_expenses"]
    total_refunds = analysis_data["total_refunds"]
    net_spending = analysis_data["net_spending"]
    highest_spending_category = analysis_data["highest_spending_category"]
    budget_status = analysis_data["budget_status"]

    # Create a system prompt that gives the chatbot context from the user's CSV data.
    system_prompt = f"""
You are Budget Buddy, a friendly and professional personal finance assistant.

You help users understand their spending based on their actual CSV transaction data.

User's transaction summary:
- Total expenses: ${total_expenses:,.2f}
- Total refunds: ${total_refunds:,.2f}
- Net spending after refunds: ${net_spending:,.2f}
- Highest spending category: {highest_spending_category}
- Budget status: {budget_status}

Your response style:
- Be supportive, practical, and easy to understand
- Give advice based on the user's actual spending data
- Focus on budgeting, spending awareness, and simple saving opportunities
- Do not give risky investment advice, legal advice, or formal financial advice
- Keep responses clear and useful for a personal finance app user
"""

    def budget_buddy_chat(user_question):
        """
        Send the user's question to the chatbot with CSV-based financial context.
        """

        full_prompt = f"""
{system_prompt}

User question:
{user_question}

Budget Buddy response:
"""

        return get_response(full_prompt)

    return budget_buddy_chat
    # Test your chatbot
budget_buddy_chat = create_finance_chat_personality(analysis)

response = budget_buddy_chat("I spend too much on coffee, what should I do?")
print(response)

Okay, let’s tackle that coffee spending! It’s fantastic that you’ve noticed it and want to make a change – that’s a really smart first step. 

Looking at your data, shopping was your highest spending category, and you’re currently within your budget, which is great. Let’s see if we can reduce that coffee cost without breaking the bank or affecting your overall spending habits. 

Here’s what we can do:

1.  **Track It Precisely:** For the next week or two, really pay attention to *how* much you're spending on coffee. Note the type of coffee (fancy latte vs. drip), and where you're getting it from (cafe vs. home). This will give us a really clear picture of the numbers.

2.  **Explore Affordable Alternatives:**
    *   **Make it at Home:** Brewing your own coffee is *significantly* cheaper. A good travel mug and some basic coffee can save you a lot. 
    *   **Look for Deals:** Many cafes have daily or weekly specials. 

3.  **Small Changes, Big Impact:** Even reducing your coffee purcha

## RAG System for Financial Documents

In [34]:
# 🤖 AI Collaboration: Document Retrieval Setup
# AI helped set up a simple RAG-style retrieval system for Budget Buddy.
# This section retrieves useful finance guidance from small text documents.

from hands_on_ai import rag

def setup_financial_rag():
    """
    Set up a simple RAG-style system for financial documents and transaction data.

    This version uses beginner-friendly Python retrieval logic:
    1. Store finance-related documents
    2. Match the user's question with relevant documents
    3. Return a helpful answer using the retrieved context

    Returns:
        function: A question-answer function for personal finance questions
    """

    financial_documents = {
        "budgeting": """
        A good budgeting strategy is to track spending regularly, compare expenses
        against a planned budget, and separate spending into needs, wants, and savings.
        """,

        "spending": """
        If a user overspends in one area, they should set a clear spending limit,
        review small purchases, and look for cheaper alternatives.
        """,

        "transaction_summary": """
        Transaction summaries help users understand total expenses, refunds,
        net spending, and which categories cost the most.
        """,

        "category_advice": """
        The highest spending category is useful because it shows where the user
        may have the biggest opportunity to save money.
        """,

        "entertainment": """
        If a user overspends on entertainment, they can set a monthly entertainment
        budget, choose free or low-cost activities, reduce impulse purchases,
        and cancel unused subscriptions.
        """
    }

    def financial_rag_question(user_question):
        """
        Retrieve relevant finance information and answer the user's question.
        """

        question_lower = user_question.lower()
        retrieved_context = []

        # Simple keyword-based retrieval
        for topic, document in financial_documents.items():
            if topic in question_lower:
                retrieved_context.append(document)

        # Extra matching for common finance words
        if "overspend" in question_lower or "spend too much" in question_lower:
            retrieved_context.append(financial_documents["spending"])

        if "entertainment" in question_lower:
            retrieved_context.append(financial_documents["entertainment"])

        if "budget" in question_lower:
            retrieved_context.append(financial_documents["budgeting"])

        # If no specific document matches, use general budgeting advice
        if not retrieved_context:
            retrieved_context.append(financial_documents["budgeting"])

        context = "\n".join(retrieved_context)

        answer = f"""
Budget Buddy found the following relevant financial guidance:

{context}

Based on this, my advice is to set a clear spending limit, review your recent
transactions, and focus first on the category where you spend the most. Small
changes can make your budget easier to manage over time.
"""

        return answer


    return financial_rag_question


# Test your RAG system
financial_rag = setup_financial_rag()

answer = financial_rag(
    "What's a good budgeting strategy for someone who overspends on entertainment?"
)

print(answer)


Budget Buddy found the following relevant financial guidance:


        A good budgeting strategy is to track spending regularly, compare expenses
        against a planned budget, and separate spending into needs, wants, and savings.
        

        If a user overspends on entertainment, they can set a monthly entertainment
        budget, choose free or low-cost activities, reduce impulse purchases,
        and cancel unused subscriptions.
        

        If a user overspends in one area, they should set a clear spending limit,
        review small purchases, and look for cheaper alternatives.
        

        If a user overspends on entertainment, they can set a monthly entertainment
        budget, choose free or low-cost activities, reduce impulse purchases,
        and cancel unused subscriptions.
        

        A good budgeting strategy is to track spending regularly, compare expenses
        against a planned budget, and separate spending into needs, wants, and savings

## Custom Financial Tools

In [35]:
# 🤖 AI Collaboration: Custom Tool Development
# AI helped create a useful financial calculator for Budget Buddy.
# This tool helps users estimate how long it will take to reach a savings goal.

from hands_on_ai import agent

def create_savings_calculator_tool():
    """
    Create a custom savings goal calculator as an agent tool.

    The calculator takes the user's current savings, monthly contribution,
    and target amount, then calculates how many months are needed to reach
    the goal.

    Returns:
        function: A savings calculator function
    """

    def savings_calculator(current_savings, monthly_contribution, target_amount):
        """
        Calculate the time needed to reach a savings goal.

        Args:
            current_savings: Amount the user has already saved
            monthly_contribution: Amount the user can save each month
            target_amount: The user's savings goal

        Returns:
            str: User-friendly savings goal result
        """

        # Input validation
        if current_savings < 0:
            return "Current savings cannot be negative."

        if monthly_contribution <= 0:
            return "Monthly contribution must be greater than zero."

        if target_amount <= 0:
            return "Target amount must be greater than zero."

        if current_savings >= target_amount:
            return (
                "Great news! You have already reached your savings goal. "
                f"Current savings: ${current_savings:,.2f}, "
                f"Target amount: ${target_amount:,.2f}."
            )

        # Calculate remaining amount needed
        remaining_amount = target_amount - current_savings

        # Calculate months needed
        months_needed = remaining_amount / monthly_contribution

        # Round up because a partial month still means another month of saving
        months_needed = int(months_needed) if months_needed.is_integer() else int(months_needed) + 1

        years = months_needed // 12
        months = months_needed % 12

        # Format time result
        if years > 0 and months > 0:
            time_message = f"{years} year(s) and {months} month(s)"
        elif years > 0:
            time_message = f"{years} year(s)"
        else:
            time_message = f"{months} month(s)"

        return (
            "Savings Goal Result:\n"
            f"- Current savings: ${current_savings:,.2f}\n"
            f"- Monthly contribution: ${monthly_contribution:,.2f}\n"
            f"- Target amount: ${target_amount:,.2f}\n"
            f"- Remaining amount: ${remaining_amount:,.2f}\n"
            f"- Estimated time to reach goal: {time_message}\n\n"
            "Budget Buddy tip: Try setting up an automatic monthly transfer "
            "so saving becomes easier and more consistent."
        )

    return savings_calculator


# Create the savings calculator tool
savings_calculator = create_savings_calculator_tool()

# Test the custom financial tool
result = savings_calculator(
    current_savings=500,
    monthly_contribution=200,
    target_amount=2000
)

print(result)


# Register your tool with the agent system
# agent.register_tool("savings_calculator", savings_calculator)

Savings Goal Result:
- Current savings: $500.00
- Monthly contribution: $200.00
- Target amount: $2,000.00
- Remaining amount: $1,500.00
- Estimated time to reach goal: 8 month(s)

Budget Buddy tip: Try setting up an automatic monthly transfer so saving becomes easier and more consistent.


## Gradio UI Integration

In [36]:
# 🤖 AI Collaboration: Professional UI Design
# AI helped create a basic Gradio interface for Budget Buddy.
# This interface connects the main features from previous notebook sections.

import gradio as gr

def create_finance_assistant_ui():
    """
    Create a basic Gradio interface for the Budget Buddy Smart Finance Assistant.

    This UI uses the existing analysis dictionary and connects:
    - spending summary
    - Budget Buddy chatbot
    - financial advice RAG
    - savings calculator
    """

    budget_buddy_chat = create_finance_chat_personality(analysis)
    financial_rag = setup_financial_rag()
    savings_calculator = create_savings_calculator_tool()

    def show_summary():
        return f"""
Spending Summary

Total expenses: ${analysis['total_expenses']:,.2f}
Total refunds: ${analysis['total_refunds']:,.2f}
Net spending: ${analysis['net_spending']:,.2f}
Highest spending category: {analysis['highest_spending_category']}
Budget status: {analysis['budget_status']}
"""

    def chat_reply(question):
        return budget_buddy_chat(question)

    def rag_reply(question):
        return financial_rag(question)

    def savings_reply(current_savings, monthly_contribution, target_amount):
        return savings_calculator(
            current_savings,
            monthly_contribution,
            target_amount
        )

    with gr.Blocks() as demo:
        gr.Markdown("# 💰 Budget Buddy Smart Finance Assistant")
        gr.Markdown("A simple finance assistant built with Python and Gradio.")

        with gr.Tab("Spending Summary"):
            summary_button = gr.Button("Show Summary")
            summary_output = gr.Textbox(label="Summary", lines=7)

            summary_button.click(
                fn=show_summary,
                inputs=None,
                outputs=summary_output
            )

        with gr.Tab("Chatbot"):
            question = gr.Textbox(
                label="Ask Budget Buddy",
                placeholder="I spend too much on coffee, what should I do?"
            )
            chat_button = gr.Button("Ask")
            chat_output = gr.Textbox(label="Response", lines=7)

            chat_button.click(
                fn=chat_reply,
                inputs=question,
                outputs=chat_output
            )

        with gr.Tab("Financial Advice"):
            rag_question = gr.Textbox(
                label="Ask a finance question",
                placeholder="What's a good budgeting strategy for entertainment?"
            )
            rag_button = gr.Button("Get Advice")
            rag_output = gr.Textbox(label="Advice", lines=7)

            rag_button.click(
                fn=rag_reply,
                inputs=rag_question,
                outputs=rag_output
            )

        with gr.Tab("Savings Calculator"):
            current_savings = gr.Number(label="Current savings", value=500)
            monthly_contribution = gr.Number(label="Monthly contribution", value=200)
            target_amount = gr.Number(label="Target amount", value=2000)

            savings_button = gr.Button("Calculate")
            savings_output = gr.Textbox(label="Result", lines=7)

            savings_button.click(
                fn=savings_reply,
                inputs=[current_savings, monthly_contribution, target_amount],
                outputs=savings_output
            )

    return demo


# Launch inside Google Colab only
demo = create_finance_assistant_ui()
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bb1dbb672f4f05872f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---

# 🧪 STEP 6: Test with a Variety of Data

**🔍 Comprehensive Testing Strategy**



## Foundation Function Tests

In [39]:
# 🤖 AI Collaboration: Foundation Function Tests
# AI helped create test cases for the core Budget Buddy functions.
# These tests use the existing sample transaction CSV file.

import pandas as pd

def test_existing_csv_data():
    """
    Test the existing CSV file already included in the project.
    """
    print("🧪 Testing existing CSV transaction data...")

    df = pd.read_csv("sample_transactions.csv")

    result = analyze_spending_patterns(df, monthly_budget=1000)

    assert "total_expenses" in result
    assert "total_refunds" in result
    assert "net_spending" in result
    assert "highest_spending_category" in result
    assert "budget_status" in result

    assert result["total_expenses"] >= 0
    assert result["total_refunds"] >= 0
    assert result["net_spending"] == result["total_expenses"] - result["total_refunds"]

    print("✅ Existing CSV data test passed.")


def test_refunds_and_large_amounts():
    """
    Test refunds and large transaction amounts.
    """
    print("🧪 Testing refunds and large amounts...")

    df = pd.DataFrame({
        "Date": ["2026-05-01", "2026-05-02", "2026-05-03"],
        "Description": ["Laptop", "Partial refund", "Groceries"],
        "Category": ["Shopping", "Refund", "Food"],
        "Amount": [2000.00, -500.00, 100.00]
    })

    result = analyze_spending_patterns(df, monthly_budget=5000)

    assert result["total_expenses"] == 2100.00
    assert result["total_refunds"] == 500.00
    assert result["net_spending"] == 1600.00
    assert result["highest_spending_category"] == "Shopping"

    print("✅ Refunds and large amounts test passed.")


def test_missing_data():
    """
    Test missing values in transaction data.
    """
    print("🧪 Testing missing data...")

    df = pd.DataFrame({
        "Date": ["2026-05-01", "2026-05-02", "2026-05-03"],
        "Description": ["Coffee", None, "Dinner"],
        "Category": ["Food", "Food", None],
        "Amount": [6.00, 12.00, 25.00]
    })

    result = analyze_spending_patterns(df, monthly_budget=100)

    assert "total_expenses" in result
    assert "net_spending" in result
    assert result["total_expenses"] >= 0

    print("✅ Missing data test passed.")


def test_invalid_data_type():
    """
    Test invalid input that is not a pandas DataFrame.
    """
    print("🧪 Testing invalid data type...")

    invalid_data = "This is not a DataFrame"

    try:
        analyze_spending_patterns(invalid_data, monthly_budget=100)
        assert False, "The function should raise an error for invalid data."
    except Exception:
        assert True

    print("✅ Invalid data type test passed.")


def test_business_logic_validation():
    """
    Test spending calculation business logic.
    """
    print("🧪 Testing business logic validation...")

    df = pd.DataFrame({
        "Date": ["2026-05-01", "2026-05-02", "2026-05-03"],
        "Description": ["Coffee", "Restaurant", "Bus"],
        "Category": ["Food", "Food", "Transport"],
        "Amount": [10.00, 40.00, 15.00]
    })

    result = analyze_spending_patterns(df, monthly_budget=100)

    assert result["total_expenses"] == 65.00
    assert result["total_refunds"] == 0.00
    assert result["net_spending"] == 65.00
    assert result["highest_spending_category"] == "Food"
    assert "budget_status" in result

    print("✅ Business logic validation test passed.")


def run_foundation_tests():
    """
    Run all foundation function tests.
    """
    print("🧪 Running Budget Buddy foundation tests...\n")

    test_existing_csv_data()
    test_refunds_and_large_amounts()
    test_missing_data()
    test_invalid_data_type()
    test_business_logic_validation()

    print("\n✅ All foundation function tests completed successfully!")


# Run foundation tests
run_foundation_tests()

🧪 Running Budget Buddy foundation tests...

🧪 Testing existing CSV transaction data...
✅ Existing CSV data test passed.
🧪 Testing refunds and large amounts...
✅ Refunds and large amounts test passed.
🧪 Testing missing data...
✅ Missing data test passed.
🧪 Testing invalid data type...
✅ Invalid data type test passed.
🧪 Testing business logic validation...
✅ Business logic validation test passed.

✅ All foundation function tests completed successfully!


## Advanced Integration Tests

In [41]:
# 🤖 AI Collaboration: Integration Testing
# AI helped create tests for the complete Budget Buddy workflow.
# These tests check that the analysis, chatbot, RAG system, and savings tool work together.

import pandas as pd

def test_full_workflow():
    """
    Test the complete workflow from CSV data to final recommendations.
    """
    print("🧪 Testing complete workflow integration...")

    # 1. Load existing sample CSV data
    df = pd.read_csv("sample_transactions.csv")

    assert isinstance(df, pd.DataFrame)
    assert len(df) > 0

    # 2. Run complete spending analysis pipeline
    analysis = analyze_spending_patterns(df, monthly_budget=1000)

    assert "total_expenses" in analysis
    assert "total_refunds" in analysis
    assert "net_spending" in analysis
    assert "highest_spending_category" in analysis
    assert "budget_status" in analysis

    # 3. Create Budget Buddy chatbot from the analysis results
    budget_buddy_chat = create_finance_chat_personality(analysis)

    chat_response = budget_buddy_chat(
        "What should I do to improve my spending habits?"
    )

    assert chat_response is not None
    assert isinstance(chat_response, str)
    assert len(chat_response) > 0

    # 4. Test RAG-style financial document retrieval
    financial_rag = setup_financial_rag()

    rag_response = financial_rag(
        "What's a good budgeting strategy for entertainment spending?"
    )

    assert rag_response is not None
    assert isinstance(rag_response, str)
    assert len(rag_response) > 0

    # 5. Test custom savings calculator tool
    savings_calculator = create_savings_calculator_tool()

    savings_result = savings_calculator(
        current_savings=500,
        monthly_contribution=200,
        target_amount=2000
    )

    assert savings_result is not None
    assert isinstance(savings_result, str)
    assert "Savings Goal Result" in savings_result

    print("✅ Full workflow integration test passed.")


def test_error_handling():
    """
    Test simple error handling and user input validation.
    """
    print("🧪 Testing error handling...")

    # Test invalid transaction input
    try:
        analyze_spending_patterns("not a dataframe", monthly_budget=1000)
        assert False, "Expected an error for invalid transaction data."
    except Exception:
        assert True

    # Test missing chatbot analysis keys
    try:
        create_finance_chat_personality({"total_expenses": 100})
        assert False, "Expected an error for missing analysis values."
    except KeyError:
        assert True

    # Test savings calculator invalid input
    savings_calculator = create_savings_calculator_tool()

    result = savings_calculator(
        current_savings=500,
        monthly_contribution=0,
        target_amount=2000
    )

    assert isinstance(result, str)
    assert "Monthly contribution" in result or "greater than zero" in result

    # Test RAG system still returns advice for a general question
    financial_rag = setup_financial_rag()

    rag_result = financial_rag("How can I save more money?")

    assert isinstance(rag_result, str)
    assert len(rag_result) > 0

    print("✅ Error handling test passed.")


# Run integration tests
try:
    test_full_workflow()
    test_error_handling()
    print("✅ Integration tests completed successfully!")
except Exception as e:
    print(f"⚠️ Integration test issue: {e}")

🧪 Testing complete workflow integration...
✅ Full workflow integration test passed.
🧪 Testing error handling...
✅ Error handling test passed.
✅ Integration tests completed successfully!
